# Week 8 activity · The Tube, three ways

**LDSCI6253 Information Presentation & Data Visualisation** · Northeastern University London

In the tutorial notebook you built networks and trees step by step. Now use the same moves on new questions
about the London Underground in Zone 1. Keep the tutorial open beside this: each part says which section to
look back at.

| Part | You will… | Tutorial | Time |
|---|---|---|---|
| Warm-up | remember three ideas | – | 2 min |
| **A** | find out what one row of the data really is | §1 | 10 min |
| **B** | count **degree** two ways | §1, §4 | 10 min |
| **C** | draw the Tube as **dots and lines**, sized by how many lines stop there | §3, §4, §6a | 15 min |
| **D** | turn the Tube into a network of **lines**, drawn as a **grid** | §5, §5b | 25 min |
| **F** | choose the right picture for three readers | §0, §6 | 10 min |
| **E** | test whether the Tube is a **tree** (if you have time, or at home) | §6c, §6 | 10 min |

**How to work, in pairs.** One of you types, the other is the checker. Before each Run, the checker says what
they expect to see; after it, compare. Swap at each new letter.

- Where you see `___`, replace it with code, then run the cell (Shift + Enter). Run cells **in order**.
- After many cells a ✅ or ❌ line tells you if you are on track.
- Stuck? Open the 💡 **Hint**, then look at the tutorial section named.
- Write your answers in the ✏️ cells (double-click to type, Shift + Enter to finish).

**If you get an error**
- `NameError: name '…' is not defined`: a cell above was not run. Run the cells above, in order.
- `SyntaxError` pointing at `___`: a blank is still empty.
- `KeyError: '…'`: a column name is misspelt. Check it against the table's `.head()`.
- A chart that is empty, or has no labels: a name inside quotes is misspelt (`'lines:Q'`, `'datum.lines'`).

## 0 · Set up

Run this cell first. It loads the two tables and gives the Tube lines their proper names.

In [ ]:
import altair as alt
import pandas as pd
from pathlib import Path

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week8/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

def check(ok, good, hint):
    """Print ✅ and what you found, or ❌ and what to look at."""
    print('✅ ' + good if ok else '❌ Not yet: ' + hint)

stations = load('tube_stations')   # name, lon, lat, zone: one row per station
edges = load('tube_links')         # source, target, line, colour: one row per ...? (Part A finds out)

# Readers should see 'Hammersmith & City', not the data's code 'hammersmith-city'
NAMES = {'bakerloo': 'Bakerloo', 'central': 'Central', 'circle': 'Circle', 'district': 'District',
         'hammersmith-city': 'Hammersmith & City', 'jubilee': 'Jubilee', 'metropolitan': 'Metropolitan',
         'northern': 'Northern', 'piccadilly': 'Piccadilly', 'victoria': 'Victoria', 'waterloo-city': 'Waterloo & City'}
edges['line'] = edges['line'].map(NAMES)
edges.head()

### Warm-up (2 min, from memory, no looking)

1. What is the **degree** of a node?
2. What makes a network a **tree**?
3. Which picture answers *"is this pair linked?"* best: dots and lines, a grid, or boxes inside boxes?

✏️ *1: …  2: …  3: …*

## A · What is a row? (10 min · tutorial §1)

**A1.** How many stations are there, and how many rows are in the table of edges?

In [ ]:
print(len(stations), 'stations')
print(len(___), 'rows in edges')

**A2.** If every row were a different piece of track, each **pair of stations** would appear once. Keep only
the first row for each pair (the same `source` and the same `target`) and count again. Tutorial §1 shows how,
with `drop_duplicates`.

In [ ]:
pairs = edges.drop_duplicates(subset=[___])
print(len(pairs), 'different pairs of stations')
check(len(pairs) == 90, 'right: 90 different pairs', 'name BOTH columns, each in quotes, as in tutorial §1')

<details><summary>💡 <b>Hint</b></summary>

`subset=` takes a **list of column names, each in quotes**, like `subset=['col1', 'col2']`. Using only
`'source'` gives 45 with no error, but that keeps one row per *starting* station, not per pair.
</details>

**A3.** 130 rows, but only 90 pairs. Show the rows whose pair appears more than once, sorted so the repeats sit
together. Look at the `line` column.

In [ ]:
edges[edges.duplicated(subset=[___], keep=False)].sort_values(['source', 'target']).head(6)

✏️ *In one sentence: what does one row of `edges` stand for?*

## B · Degree, two ways (10 min · tutorial §1, §4)

In the tutorial's §4 Try it, King's Cross St. Pancras came out with degree 10. But only six lines stop there,
and only five stations are one stop away. So what was counted? Count two different things:

- **edges**: every row of `edges` that touches the station (so shared track counts once per line);
- **neighbours**: how many *different* stations are one stop away (count in `pairs`, from A2).

**B1.** Count both, the same way as tutorial §1, and put them side by side.

In [ ]:
by_edges = pd.concat([edges['source'], edges['target']]).value_counts()
by_neighbours = pd.concat([___]).value_counts()

degree = pd.DataFrame({'edges': by_edges, 'neighbours': by_neighbours})
check(degree.loc["King's Cross St. Pancras", 'neighbours'] == 5, "right: King's Cross has 5 neighbours",
      "count in pairs (from A2), not in edges")
degree.sort_values('neighbours', ascending=False).head(8)

**B2.** How many *extra* edges does each station get from shared track? Subtract one column from the other.

In [ ]:
degree['extra'] = ___
degree.sort_values('extra', ascending=False).head(5)

✏️ *Which station is busiest by edges, and which by neighbours? Why do the answers differ? Use A3.*

## C · Dots and lines: where the lines meet (15 min · tutorial §3, §4, §6a)

**C1.** Make a table of which lines serve each station (one row per station and line), then count the lines at
each station. This is like stacking `source` and `target` for degree, but keeping the line beside each name.

In [ ]:
served = pd.concat([
    edges[['source', 'line', 'colour']].rename(columns={'source': 'station'}),
    edges[['target', 'line', 'colour']].rename(columns={'target': 'station'}),
]).drop_duplicates()

# Count the rows (lines) for each station. It is named 'lines': the chart in C2 uses that name.
lines_at = served.groupby('station')['line'].___().rename('lines')
check(lines_at.max() == 6, "right: the busiest station has 6 lines", "count the lines in each station's group")
lines_at.sort_values(ascending=False).head(6)

**C2.** Draw the Tube as in tutorial §4, but **size each station by how many lines stop there**, and name the
stations with **five or more** lines (more names would pile up in the middle). The colours now have a key.

In [ ]:
dots_table = stations.merge(lines_at, left_on='name', right_index=True)   # as in tutorial §3

# The same join as the tutorial: each edge gets both of its ends
track = (edges
    .merge(stations, left_on='source', right_on='name')
    .merge(stations, left_on='target', right_on='name', suffixes=('', '2')))

where = dict(x=alt.X('lon:Q', scale=alt.Scale(zero=False), axis=None),
             y=alt.Y('lat:Q', scale=alt.Scale(zero=False), axis=None))

# TfL's own colour for each line, with a key (scale=None would use the colours but give no key)
key = edges[['line', 'colour']].drop_duplicates()
line_colours = alt.Scale(domain=key['line'].tolist(), range=key['colour'].tolist())

rails = alt.Chart(track).mark_rule(strokeWidth=3).encode(
    **where, x2='lon2:Q', y2='lat2:Q', color=alt.Color('line:N', scale=line_colours, title='Line'))
dots = alt.Chart(dots_table).mark_circle(color='white', stroke='black', opacity=1).encode(
    **where, size=alt.Size(___, title='Lines stopping here', scale=alt.Scale(domain=[1, 6]),
                           legend=alt.Legend(values=[1, 2, 3, 4, 5, 6])),
    tooltip=['name:N', 'lines:Q'])
names = alt.Chart(dots_table).mark_text(dy=-14, fontWeight='bold', fontSize=11).encode(
    **where, text='name:N').transform_filter(___)

(rails + dots + names).properties(
    width=640, height=440,
    title=alt.Title("King's Cross and Baker Street are where most lines meet", subtitle='Source: TfL Open Data')
).configure_view(stroke=None)

<details><summary>💡 <b>Hint</b></summary>

- Size, as in tutorial §3's *size each dot by its degree*: the column's **name in quotes** with its type, `'lines:Q'`.
- The filter, as in tutorial §6a: `'datum.lines >= 5'`. `datum` means *this row*. Not `dots_table['lines'] >= 5`
  (that is pandas, not a chart filter), and not `'lines >= 5'` (it needs `datum.`).
</details>

✏️ *What does position mean in this picture? What does size mean? Name one thing this picture is bad at showing.*

## D · A network of lines, as a grid (25 min · tutorial §5, §5b)

Turn the question round: now the **nodes are the 11 lines**, and two lines are linked when they **share a
station**. The more stations they share, the stronger the link: its **weight**. This is exactly tutorial §5b,
with lines instead of people and stations instead of clubs.

**D1a.** Merge `served` with itself on the column two lines have in common, and look at the first rows.

In [ ]:
both = served.merge(served, on=___)
check('line_x' in both.columns and (both['station'] == "King's Cross St. Pancras").sum() == 36,
      'right: every pair of lines at each station', 'merge on the column two LINES share (tutorial §5b merged on club)')
both.head(8)

<details><summary>💡 <b>Hint</b></summary>

In tutorial §5b, people were linked when they shared a **club**, so the merge was `on='club'`. Here lines are
linked when they share a … ? Merging on `'line'` instead pairs each station with every other station on the
same line: a different network.
</details>

✏️ *What does one row of `both` mean now?*

**D1b.** Drop each line paired with itself, then count the stations each pair of lines shares.

In [ ]:
both = both[both['line_x'] ___ both['line_y']]
shared = both.groupby(['line_x', 'line_y']).size().reset_index(name='stations')
check(shared['stations'].max() == 18, 'right: the most shared is 18 stations', "keep the rows where the two lines are NOT the same")
shared.sort_values('stations', ascending=False).head(6)

**D2.** Draw it as a **grid**, as in tutorial §5: one row and one column per line, the square's shade for how
many stations they share, and the number written in it. Rows go from the line with the most stations to the
fewest.

In [ ]:
order = served['line'].value_counts().index.tolist()   # most stations first, as in tutorial §5

cells = alt.Chart(shared).encode(
    x=alt.X('line_x:N', sort=order, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('line_y:N', sort=order, title=None))
squares = cells.mark_rect().encode(
    color=alt.Color(___, scale=alt.Scale(scheme='blues'), title='Stations shared'),
    tooltip=['line_x:N', 'line_y:N', 'stations:Q'])
numbers = cells.mark_text(fontSize=11).encode(
    text='stations:Q',
    color=alt.condition('datum.stations > 8', alt.value('white'), alt.value('black')))   # light text on dark squares

(squares + numbers).properties(
    width=400, height=400,
    title=alt.Title('Circle and District share the most stations', subtitle='Source: TfL Open Data')
).configure_view(stroke=None)

<details><summary>💡 <b>Hint</b></summary>

As `'scenes:Q'` in tutorial §5: the weight column's name, with `:Q` because it is a number. `'stations:N'` also
draws, but as separate colours with no order.
</details>

To compare, here is the same network as **dots and lines**: the 11 lines on a circle, joined when they share a
station, thicker for more. (Just run it.)

In [ ]:
import math
ring = pd.DataFrame({'line': order})
ring['x'] = [math.cos(2 * math.pi * i / len(order)) for i in range(len(order))]
ring['y'] = [math.sin(2 * math.pi * i / len(order)) for i in range(len(order))]
spokes = (shared[shared['line_x'] < shared['line_y']]   # each pair once
    .merge(ring, left_on='line_x', right_on='line')
    .merge(ring, left_on='line_y', right_on='line', suffixes=('', '2')))
at = dict(x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[-1.4, 1.4])),
          y=alt.Y('y:Q', axis=None, scale=alt.Scale(domain=[-1.25, 1.25])))
alt.layer(
    alt.Chart(spokes).mark_rule(color='steelblue', opacity=0.6).encode(
        **at, x2='x2:Q', y2='y2:Q', strokeWidth=alt.StrokeWidth('stations:Q', title='Stations shared', scale=alt.Scale(range=[1, 10]))),
    alt.Chart(ring).mark_circle(size=200, color='black', opacity=1).encode(**at),
    alt.Chart(ring).mark_text(dy=-16, fontWeight='bold').encode(**at, text='line:N'),
).properties(width=460, height=420, title='The same network as dots and lines').configure_view(stroke=None)

✏️ *Which two lines share the most stations? Which picture lets you read the Circle line's links more easily, and why?*

## F · Choose and justify (10 min · tutorial §0, §6)

For each reader, choose **dots and lines**, **a grid**, or **boxes / bars inside each other**. Give one reason,
and say why **not** one of the other two. Use the pictures you made (the council's question is about tutorial §6).

1. A visitor at Paddington asks: *"Can I get to Bank, and where do I change?"*
2. TfL planners ask: *"Which line shares the most stations with the Circle line?"*
3. A council asks: *"Which areas of London have the most people, and which boroughs make them up?"*

✏️ *1: … because … , not … because …  2: …  3: …*

**F2 (extension).** Improve one of your pictures for the visitor or the planner and explain your change in one
sentence. A starter, which draws the Central line thick on top of the map from C2, is below.

In [ ]:
central = track[track['line'] == 'Central']
thick = alt.Chart(central).mark_rule(strokeWidth=8, color='#E32017').encode(**where, x2='lon2:Q', y2='lat2:Q')
(rails + thick + dots + names).properties(width=640, height=440, title='Change your title to say what to see')

## E · Is the Tube a tree? (10 min · tutorial §6c and *When does a tree lie?*) · if you have time, or at home

A **tree** needs every child to have exactly **one** parent. Suppose we try Tube → line → station.

**E1.** How many stations have more than one line? (Use `lines_at` from C1. Tutorial §1: adding up `True`s counts them.)

In [ ]:
print((lines_at ___).sum(), 'of', len(lines_at), 'stations are on more than one line')
lines_at.value_counts().sort_index().rename('stations')   # how many stations have 1, 2, 3 ... lines

**E2.** Pretend it is a tree anyway, and draw an icicle as in tutorial §6c: one bar per level, **counting**
stations (tutorial §5b showed how to count rows in a chart). The red line marks how many stations there really are.

In [ ]:
rows = pd.concat([
    served.assign(level='1 Tube', name='Tube', colour='#888888'),
    served.assign(level='2 Line', name=served['line']),
    served.assign(level='3 Station', name=served['station']),
])
rows['path'] = rows['line'] + rows['station']   # keep stations under their line, as in tutorial §6c

bars = alt.Chart(rows).mark_bar(stroke='white').encode(
    y=alt.Y('level:O', title=None),
    x=alt.X(___, title='Stations counted'),
    detail='name:N',
    order=alt.Order('min(path):N'),
    color=alt.Color('colour:N', scale=None),
    tooltip=['name:N'])
real = alt.Chart(pd.DataFrame({'n': [len(stations)]}))
truth = real.mark_rule(color='red', strokeWidth=3).encode(x='n:Q') + real.mark_text(
    color='red', align='left', dx=4, dy=-62, fontWeight='bold').encode(x='n:Q', text=alt.value('65 real stations'))

(bars + truth).properties(
    width=640, height=180,
    title=alt.Title('The Tube forced into a tree counts 139 stations', subtitle='Source: TfL Open Data'))

<details><summary>💡 <b>Hint</b></summary>

`'count():Q'` counts rows, as in tutorial §5b. `'sum(...)'` adds up a column, and `'station:Q'` treats names as
numbers: both draw an empty chart with no error.
</details>

✏️ *Is the Tube a tree? What goes wrong in the icicle when we pretend it is?*